In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage

In [16]:
generator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
optimizer_llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from pydantic import BaseModel,Field

class PostEvaluation(BaseModel):
    evaluation: Literal['approved','needs_improvement'] = Field(...,description="Final Evaluation")
    feedback: str = Field(...,description="Constructive feedback for the post")

In [4]:
structured_evaluator_llm = evaluator_llm.with_structured_output(PostEvaluation)


In [5]:
class PostState(TypedDict):
    topic:str
    tweet:str
    evaluation: Literal["approved","needs_improvement"]
    feedback:str
    iteration:int
    max_iteration:int

In [6]:
def generate_Linkedin_Post(state:PostState):
    # prompt
    messages = [
    SystemMessage(
        content="You are a professional senior full-stack software engineer."
    ),
    HumanMessage(
        content=f"""
Write a minimal, original LinkedIn post on the topic: "{state['topic']}".

Rules:
- Do not use a question-answer format.
- Maximum 500 characters.
- Use observational humor and professional references.
- Use simple, everyday English.
- This is version {state['iteration'] + 1}.
"""
    ),

    ]

# send generator llm
    response = generator_llm.invoke(messages).content
    # return response
    return {"tweet":response}


In [7]:
def evaluate_post(state:PostState):
    messages = [
    SystemMessage(
        content="You are an expert LinkedIn content strategist and editor."
    ),
    HumanMessage(
        content=f"""
Evaluate the following LinkedIn post.

Post:
{state['tweet']}

Scoring Criteria (0-10):
1. Originality
2. Clarity
3. Engagement Potential
4. Professional Relevance
5. Humor Quality
6. Readability
7. Authenticity
8. LinkedIn Suitability

Rules:
- Be strict and objective.
- Penalize generic AI-style writing.
- Penalize clichés, buzzwords, and overused phrases.
- Reward unique observations and authentic voice.
- Consider whether the post sounds like it was written by a real software engineer.
- Explain each score in 1-2 sentences.

Auto-reject if:
- It is written in question and format
- It exceed 500 characters
- It reads like a boring way

Return your response in this exact format:

Overall Score: X/10
evaluation: 'approved' or "need_improvement"
feedback: One Paragraph explaining the strengths and weaknesses

"""
    ),
]
    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation':response.evaluation, "feedback":response.feedback}


In [8]:
def optimised_post(state:PostState):
    messages = [
    SystemMessage(
        content="You are an expert LinkedIn content editor."
    ),
    HumanMessage(
        content=f"""
Improve this LinkedIn post based on the feedback.

Original Post:
{state['tweet']}

Topic: "{state['topic']}"

Feedback:
{state['feedback']}

Rewrite it as
Rules:
- Keep the core idea unchanged.
- Address the feedback.
- Make it more engaging and authentic.
- Use simple everyday English.
- Keep it under 500 characters.
- Return only the improved post.
"""
    ),
]
    
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration']+1

    return {"tweet": response, "iteration": iteration}

In [9]:
def route_evaluation(state:PostState):
    if state['evaluation'] == "approved" or state['iteration'] >= state['max_iteration']:
        return "approved"
    else:
        return "needs_improvement"

In [10]:
graph = StateGraph(PostState)

graph.add_node("generate",generate_Linkedin_Post)
graph.add_node("evaluate",evaluate_post)
graph.add_node("optimize",optimised_post)

graph.add_edge(START,"generate")
graph.add_edge("generate","evaluate")

graph.add_conditional_edges("evaluate",route_evaluation,{'approved':END,'needs_improvement':"optimize"})
graph.add_edge("optimize","evaluate")

workflow = graph.compile()


In [17]:
initial_state = {
    "topic": "langchain for agentic ai",
    "iteration":1,
    "max_iteration":5
}
workflow.invoke(initial_state)

{'topic': 'langchain for agentic ai',
 'tweet': "LangChain is revolutionizing agentic AI by moving beyond basic chatbots to create systems that can genuinely think and adapt in real-time. Imagine chatting with an AI that not only answers your questions but also understands the nuances of the conversation. For example, if you're discussing a technical issue, it can provide tailored solutions based on previous interactions. While there are still challenges with context understanding, the advancements with LangChain are helping us build more intuitive and responsive AI systems. Let's dive into this transformation together!",
 'evaluation': 'approved',
 'feedback': 'The post does a commendable job of discussing the advancements in agentic AI with LangChain, highlighting its real-time adaptability and nuanced conversation understanding. It maintains a professional tone suitable for LinkedIn, demonstrating clarity and engagement potential through its forward-thinking perspective. However, it